# Phase 1 — Exploration et vérification de la randomisation

**Question de cette phase** : avant de mesurer quoi que ce soit, peut-on faire confiance à l'expérience ?
Si l'assignation aux groupes est vraiment aléatoire, les trois groupes sont statistiquement identiques *avant* l'envoi des e-mails — et toute différence observée *après* est attribuable à l'e-mail (effet **causal**). C'est ce qu'on vérifie ici.

## 1. Charger les données

`pd.read_csv` lit le fichier en `DataFrame` : une ligne par client, une colonne par variable.

In [1]:
import pandas as pd

df = pd.read_csv("../data/hillstrom.csv")
df.head()

,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


## 2. Structure : dimensions, types, valeurs manquantes

- `shape` : (nombre de lignes, nombre de colonnes).
- `dtypes` : `int64`/`float64` = numérique, `str` = texte (catégoriel ; `object` dans les versions de pandas antérieures à 3.0).
- `isna().sum()` : nombre de valeurs manquantes par colonne.

In [2]:
print(df.shape)
pd.DataFrame({"type": df.dtypes, "manquants": df.isna().sum(), "valeurs distinctes": df.nunique()})

(64000, 12)


,type,manquants,valeurs distinctes
recency,int64,0,12
history_segment,str,0,7
history,float64,0,34833
mens,int64,0,2
womens,int64,0,2
zip_code,str,0,3
newbie,int64,0,2
channel,str,0,3
segment,str,0,3
visit,int64,0,2


## 3. Taille des groupes de traitement

`segment` est la variable de **traitement**. Le protocole annonce une répartition 1/3 – 1/3 – 1/3 : on vérifie d'abord à l'œil.

In [3]:
df["segment"].value_counts()

segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64

## 4. Test du χ² d'ajustement : y a-t-il un *sample ratio mismatch* ?

**Idée** : on compare les effectifs **observés** aux effectifs **attendus** si l'assignation suivait exactement le protocole (1/3 chacun, soit 64 000 / 3 ≈ 21 333 par groupe).

**Statistique** : χ² = Σ (observé − attendu)² / attendu. Elle vaut 0 si tout colle parfaitement et grandit avec les écarts.

**Hypothèse nulle H₀** : la vraie répartition est 1/3 – 1/3 – 1/3, et les écarts observés ne viennent que du hasard du tirage.

**p-value** : la probabilité d'obtenir un χ² au moins aussi grand que celui observé *si H₀ est vraie*. Une p-value très petite (en pratique, pour un contrôle de *sample ratio mismatch*, on prend souvent un seuil strict comme 0,001) signale un problème dans l'assignation : bug de tirage, clients perdus dans un groupe, etc. Dans ce cas, aucune conclusion causale ne serait fiable.

Avec 3 groupes, on a 3 − 1 = **2 degrés de liberté** (une fois deux effectifs fixés, le troisième est imposé par le total).

In [4]:
from scipy import stats

observes = df["segment"].value_counts()
attendus = [len(df) / 3] * 3

chi2, p_value = stats.chisquare(f_obs=observes, f_exp=attendus)

print(pd.DataFrame({"observé": observes, "attendu": attendus, "écart": observes - attendus}).round(1))
print()
print(f"χ² = {chi2:.3f}   (2 degrés de liberté)")
print(f"p-value = {p_value:.3f}")

               observé  attendu  écart
segment                               
Womens E-Mail    21387  21333.3   53.7
Mens E-Mail      21307  21333.3  -26.3
No E-Mail        21306  21333.3  -27.3

χ² = 0.203   (2 degrés de liberté)
p-value = 0.904


**Interprétation** : χ² ≈ 0,2 et p ≈ 0,90. Si l'assignation était parfaitement 1/3 – 1/3 – 1/3, on observerait des écarts au moins aussi grands que ceux-ci dans environ 90 % des expériences. Le plus gros écart (+54 clients pour l'e-mail Femmes) représente 0,25 % de l'effectif attendu, ce qui relève du bruit normal d'un tirage aléatoire.

➡️ **Aucun *sample ratio mismatch*** : la répartition est compatible avec le protocole annoncé.

⚠️ Une p-value élevée ne *prouve* pas que H₀ est vraie ; elle dit seulement que les données ne fournissent aucune raison d'en douter. Ce test ne vérifie que les **tailles** de groupes, pas leur **composition** : c'est l'objet de l'étape suivante.